In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import hashlib

import torch

REPO_URL = "https://github.com/MRdrissi/industrial-defect-ai.git"
BRANCH = "feature/us-03-04-cae-hazelnut"

REPO_DIR = Path("/kaggle/working/industrial-defect-ai")

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        [
            "git", "clone",
            "--branch", BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO_DIR)
        ],
        check=True
    )

COMMIT_SHA = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True
).strip()

sys.path.insert(0, str(REPO_DIR))

print("Code version:", COMMIT_SHA)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "Activer un GPU dans Kaggle."

DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import csv

experiment_path = (
        REPO_DIR / "experiments/configs/cae_hazelnut.json"
)

with experiment_path.open(encoding="utf-8") as f:
    experiment = json.load(f)

architecture_path = (
        REPO_DIR / experiment["architecture_config"]
)

with architecture_path.open(encoding="utf-8") as f:
    architecture = json.load(f)["config"]

manifest_path = (
        REPO_DIR / experiment["dataset"]["manifest"]
)

manifest_sha256 = hashlib.sha256(
    manifest_path.read_bytes()
).hexdigest()

EXPECTED_MANIFEST_SHA = (
    "8f0e0d5b5bb767cd2d2e6f8a0ab10f65a959a97a38bb14bc5d9aae1d30c81fb6"
)

assert manifest_sha256 == EXPECTED_MANIFEST_SHA

with manifest_path.open(
        encoding="utf-8", newline=""
) as f:
    rows = list(csv.DictReader(f))

train_rows = [
    row for row in rows
    if row["category"] == "hazelnut"
       and row["split"] == "train"
]

val_rows = [
    row for row in rows
    if row["category"] == "hazelnut"
       and row["split"] == "val"
]

assert len(train_rows) == 313
assert len(val_rows) == 78

assert all(
    row["label"] == "0" and row["defect_type"] == "good"
    for row in train_rows + val_rows
)

train_paths = {row["image_path"] for row in train_rows}
val_paths = {row["image_path"] for row in val_rows}

assert train_paths.isdisjoint(val_paths)

print("Experiment:", experiment["experiment_name"])
print("Manifest SHA256:", manifest_sha256)
print("Train:", len(train_rows))
print("Validation:", len(val_rows))
print("Train/Val disjoint:", True)
print("Dataset protocol: OK")


In [ ]:
from src.data.mvtec_dataset import MVTecDataset
from src.data.transforms import MVTecTransform
from src.data.dataloaders import create_dataloader

# Identifier la racine réelle du dataset attaché
good_dirs = list(
    Path("/kaggle/input").rglob("hazelnut/train/good")
)

assert len(good_dirs) == 1, (
    f"Racine Hazelnut ambiguë ou absente : {good_dirs}"
)

DATASET_ROOT = good_dirs[0].parents[2]

print("MVTec root:", DATASET_ROOT)

transform = MVTecTransform(
    size=tuple(architecture["image_size"]),
    profile="cae"
)

train_dataset = MVTecDataset(
    dataset_root=DATASET_ROOT,
    manifest_path=manifest_path,
    category="hazelnut",
    split="train",
    transform=transform
)

val_dataset = MVTecDataset(
    dataset_root=DATASET_ROOT,
    manifest_path=manifest_path,
    category="hazelnut",
    split="val",
    transform=transform
)

assert len(train_dataset) == 313
assert len(val_dataset) == 78

sample = train_dataset[0]

assert sample["image"].shape == (3, 256, 256)
assert sample["image"].dtype == torch.float32
assert 0 <= sample["image"].min().item()
assert sample["image"].max().item() <= 1
assert sample["label"] == 0

print("Sample shape:", tuple(sample["image"].shape))
print("Sample range:", (
    sample["image"].min().item(),
    sample["image"].max().item()
))
print("Hazelnut datasets: OK")


In [ ]:
from src.models.convolutional_autoencoder import (
    CAEConfig,
    ConvolutionalAutoencoder
)

training = experiment["training"]

config = CAEConfig(
    input_channels=architecture["input_channels"],
    image_size=tuple(architecture["image_size"]),
    encoder_channels=tuple(
        architecture["encoder_channels"]
    ),
    latent_channels=architecture["latent_channels"],
    seed=training["seed"]
)

torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)

model = ConvolutionalAutoencoder(config).to(DEVICE)

parameter_count = sum(
    p.numel() for p in model.parameters()
)

assert parameter_count == 855619

model.eval()

with torch.inference_mode():
    image = sample["image"].unsqueeze(0).to(DEVICE)
    reconstruction = model(image)

assert reconstruction.shape == image.shape
assert torch.isfinite(reconstruction).all().item()

print("Parameters:", parameter_count)
print("Input:", tuple(image.shape))
print("Output:", tuple(reconstruction.shape))
print("CAE GPU forward: OK")
